# Tutorial 1 — Tokenization Deep Dive

## 0. Mental model: text is not what the model sees

A language model never sees the string `"unbelievable"`. It sees a sequence of integers, e.g.
`[403, 6667, 720]`. Getting from one to the other and back is the tokenizer's whole job — `tokenize`,
`convert_tokens_to_ids`, and `decode` — and every step in between is arithmetic on numbers:

![From raw text to the next predicted token: tokenize produces token strings, convert_tokens_to_ids produces input ids, the embedding layer turns ids into vectors, transformer layers produce contextual states, the LM head and softmax give next-token probabilities, argmax or sampling picks an id, and decode turns it back into text](images/tokenizer.png)

The tokenizer bookends the whole process: it is the first step in and the last step out, and everything
between is numbers. Note the final box — the predicted token is appended to the input and the loop runs
again, one token at a time. That is generation.

The **tokenizer** is a separate, trained artifact from the model. It has its own vocabulary file(s) and
its own training algorithm, and it is normally trained **once** and then frozen for the lifetime of a
model family. This matters practically: a model checkpoint and its tokenizer must always be loaded as a
*pair* (`AutoTokenizer.from_pretrained("same-repo-as-the-model")`) — mixing tokenizers from different
models silently produces garbage, because the integer ids mean different things to different models.

There are three traditional families of tokenization, and understanding why the field converged on the
third one is the goal of the next two sections:

| Granularity | Example split of "unbelievable" | Vocabulary size | Main failure mode |
|---|---|---|---|
| **Word-level** | `["unbelievable"]` | Huge (100k+), still has gaps | Out-of-vocabulary (OOV) words |
| **Character-level** | `["u","n","b","e","l","i","e","v","a","b","l","e"]` | Tiny (~100) | Sequences become very long; hard to learn meaning per character |
| **Subword-level** | `["un", "believ", "able"]` | Moderate (16k–128k) | Occasional awkward splits, but no OOV and manageable sequence length |

Every modern LLM (BERT, GPT-2/3/4, Llama, T5, Qwen, ...) uses some form of **subword tokenization**. The
rest of this notebook builds that idea from first principles, then shows you the production tools.


## 1. Install the libraries

In [ ]:
!pip -q install -U transformers tokenizers datasets tiktoken


### Code walkthrough — installation

- **`transformers`**: provides `AutoTokenizer`, which loads any pretrained tokenizer — BPE, WordPiece, or
  Unigram — behind one interface.
- **`tokenizers`**: the fast Rust library `AutoTokenizer` uses underneath, and which you use directly in
  section 7 to **train** a tokenizer from scratch.
- **`datasets`**: supplies the small real corpus to train on.
- **`tiktoken`**: OpenAI's byte-level BPE library, used in section 6 to inspect GPT-4-style tokenization
  without a Hugging Face checkpoint.

## 2. Why not just split on whitespace?

The simplest possible tokenizer splits on spaces and punctuation. Let's measure what that choice costs
on a small corpus before dismissing it — the failure modes are easier to trust once you've seen the numbers.


In [ ]:
import re
from collections import Counter

corpus = [
    "Tokenization is the first step of every NLP pipeline.",
    "Tokenisation is the first step of every NLP pipeline.",  # British spelling
    "The pre-tokenizer splits text before subword merges are applied.",
    "unbelievable, unbelievably, believable, believer, disbelief",
    "COVID-19 vaccination rates rose in 2021.",
    "She's running the tokenizer's unit tests at 3:45pm.",
]

def whitespace_tokenize(text):
    return text.split()

def word_punct_tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text)

for text in corpus:
    print(f"{text!r}")
    print("  whitespace :", whitespace_tokenize(text))
    print("  word+punct :", word_punct_tokenize(text))
    print()

vocab = Counter(tok for text in corpus for tok in word_punct_tokenize(text.lower()))
print("Vocabulary size for this tiny 6-sentence corpus:", len(vocab))
print("Words that share a root but get NO shared representation:")
print("  ", [w for w in vocab if "believ" in w])


### Code walkthrough — what word-level tokenization costs

Two tokenizers are compared. `text.split()` is the crudest possible version: it splits on runs of
whitespace and nothing else, so `"3:45pm."` stays glued together as a single "word".

`re.findall(r"\w+|[^\w\s]", text)` is the slightly smarter one. `\w+` grabs a whole alphanumeric run,
`|` falls through to the alternative, and `[^\w\s]` matches one character that is neither word-character
nor whitespace — a single piece of punctuation. That second branch is why `"pipeline."` comes out as
`["pipeline", "."]`. The `.lower()` before counting is a normalisation choice, folding `"The"` and
`"the"` into one entry; every tokenizer has to make that decision explicitly.

**What the output shows.** `"Tokenization"` and `"Tokenisation"` become two unrelated entries despite
meaning the same thing. `"unbelievable"`, `"unbelievably"`, `"believable"`, `"believer"`, and
`"disbelief"` all share the root `believ`, but a word-level vocabulary stores five independent ids, so the
model has to learn "believe-related meaning" five separate times.

And six sentences already need dozens of entries. A vocabulary covering real English — inflections,
typos, names, numbers — would be enormous and would *still* meet unseen words at inference time. That is
the **out-of-vocabulary (OOV)** problem, and subword tokenization exists to avoid both failures at once.

### Before you run

`"internationalization"` is 20 characters, so character-level tokenization turns it into 20 tokens where a
subword tokenizer would use two or three.

That is the trade. The vocabulary is tiny and can never meet an unknown symbol, but every sequence grows
by roughly a factor of five, and a model's context window is measured in tokens, not words. Self-attention
cost grows quadratically with sequence length, so those extra tokens are paid for twice — once in context
budget, once in compute.

In [ ]:
word = "internationalization"
char_tokens = list(word)
print(char_tokens)
print("word-level length:", 1, "tokens   |   character-level length:", len(char_tokens), "tokens")


### Code walkthrough — the other extreme

`list("internationalization")` just materialises the string's characters, since a Python string is already
an iterable of them.

Character-level tokenization has a fixed vocabulary of roughly the alphabet plus punctuation and digits,
and therefore **never** has an out-of-vocabulary problem. Its cost is the sequence length above. Subword
tokenization is the compromise: a moderate vocabulary that keeps common words at roughly one token each,
splitting only rare or morphologically complex words into pieces.

## 3. Meet a real tokenizer: `AutoTokenizer`

`AutoTokenizer.from_pretrained(...)` downloads and loads the exact tokenizer that shipped with a given
model checkpoint — vocabulary file, merge rules (section 4 explains where those come from), special tokens, and all. Loading a tokenizer never
downloads model weights, so this section stays fast even without a GPU.


In [ ]:
from transformers import AutoTokenizer

bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "Tokenization isn't as simple as splitting on spaces."

tokens = bert_tok.tokenize(text)
ids = bert_tok.convert_tokens_to_ids(tokens)
decoded = bert_tok.decode(ids)

print("tokens :", tokens)
print("ids    :", ids)
print("decoded:", decoded)


### Code walkthrough — `tokenize` / `convert_tokens_to_ids` / `decode`

**`from_pretrained("bert-base-uncased")`** fetches that repository's tokenizer files (`vocab.txt`,
`tokenizer_config.json`, …) and reconstructs the exact tokenizer BERT was trained with. Any other
tokenizer would map the same word to ids BERT never learned embeddings for.

**`.tokenize(text)`** runs pre-tokenization plus subword splitting and returns token *strings*, not ids.
Notice that `"isn't"` splits into pieces, and that any subword continuing a previous one carries a `##`
prefix — BERT's WordPiece convention for "no space before this piece". `"Tokenization"` typically becomes
`["token", "##ization"]`: one common word plus a suffix seen thousands of times elsewhere.

**`.convert_tokens_to_ids(tokens)`** is a pure dictionary lookup into the vocabulary table. It does not
re-run the splitting algorithm.

**`.decode(ids)`** reverses the whole chain, re-joining `##` continuations without inserting spaces.
Round-tripping tokenize → convert → decode is a good sanity check when a tokenizer misbehaves.

In [ ]:
encoded = bert_tok(text)
print(encoded)
print()
print("input_ids     :", encoded["input_ids"])
print("as tokens     :", bert_tok.convert_ids_to_tokens(encoded["input_ids"]))
print("attention_mask:", encoded["attention_mask"])
print("token_type_ids:", encoded["token_type_ids"])


### Code walkthrough — calling the tokenizer directly

`bert_tok(text)` is the interface you actually use: tokenize, convert, and assemble in one call, returning
a dict-like `BatchEncoding` with three keys.

- **`input_ids`** — the integer ids, now with two extra ids that were **not** in the plain `.tokenize()`
  output.
- **`attention_mask`** — all `1`s here, marking every position as real content to attend to. Padding turns
  some of these into `0` in section 8.
- **`token_type_ids`** — all `0`s, marking "first segment". BERT was pretrained on *pairs* of segments
  (question + context, say), with `1`s marking the second; single-sentence input is all segment `0`.

The two extra ids decode to `[CLS]` and `[SEP]`. **`[CLS]`** is prepended to every input, and its final
hidden state serves as a summary of the sequence for sentence-level tasks. **`[SEP]`** ends a segment and
separates a pair.

They appear because `add_special_tokens=True` is the default — which is also why `input_ids` is two longer
than the `convert_tokens_to_ids(tokenize(text))` result from the previous cell.

### Exercise 3.1

Load `AutoTokenizer.from_pretrained("gpt2")` and tokenize the same `text` variable from
above. GPT-2 uses a decoder-only, causal-LM design and was never trained on the two-segment task BERT
was. Before running anything: do you expect `token_type_ids` to appear in GPT-2's output? What about
`[CLS]`/`[SEP]`-equivalent special tokens? Then check your prediction with code.


## 4. How BPE builds a vocabulary

`bert_tok.tokenize(...)` felt like magic in the previous section. The mechanism behind it is short enough
to state in four steps.

**The BPE training algorithm, in words:**

1. Start with every word split into individual characters, plus an end-of-word marker so the model can
   tell `"est"` at the end of `"newest"` apart from `"est"` at the start of a word.
2. Count every adjacent **pair** of symbols across the whole corpus.
3. Merge the single most frequent pair into one new symbol, and record that merge rule.
4. Repeat steps 2–3 for a fixed number of merges — that count is the main knob controlling final
   vocabulary size.

Nothing here is linguistic. Pure frequency counting is what discovers that `er` at the end of a word is a
recurring pattern worth its own symbol, and the same loop scaled to billions of words and tens of thousands
of merges is what produced GPT-2's tokenizer.

Two properties follow from step 1, and both matter later. Because the algorithm can always fall back to
individual characters, **every input can be tokenized** — there is no out-of-vocabulary case, only more
splitting than usual for unfamiliar text. And because merges are applied in the order they were learned, a
merge learned early outranks one learned late.

The learned list of merge rules, applied greedily in that order, *is* the tokenizer.

## 5. BPE vs. WordPiece vs. Unigram: same idea, different scoring rule

All three families you'll meet in practice merge or split *subwords* rather than characters or whole
words; they differ in **how they decide which merge/split to make**:

| Algorithm | Merge/split criterion | Used by |
|---|---|---|
| **BPE** | Merge the most *frequent* adjacent pair | GPT-2, GPT-4 (byte-level variant), RoBERTa |
| **WordPiece** | Merge the pair that most increases the *likelihood* of the training data under a unigram language model — frequency divided by the frequencies of its parts, roughly preferring pairs whose parts are individually rare but which co-occur a lot | BERT, DistilBERT, Electra |
| **Unigram (SentencePiece)** | Start from a *large* candidate vocabulary and iteratively *remove* the pieces that hurt a probabilistic model of the corpus least, until reaching the target size — the reverse direction from BPE/WordPiece | T5, ALBERT, XLNet, Llama, Gemma |

Let's tokenize the same sentence with one tokenizer from each family and compare the actual output.


In [ ]:
from transformers import AutoTokenizer

sentence = "The unbelievably fast transformer tokenizes internationalization."

tokenizer_families = {
    "gpt2 (byte-level BPE)": "gpt2",
    "bert-base-uncased (WordPiece)": "bert-base-uncased",
    "xlnet-base-cased (Unigram/SentencePiece)": "xlnet-base-cased",
}

for label, checkpoint in tokenizer_families.items():
    tok = AutoTokenizer.from_pretrained(checkpoint)
    pieces = tok.tokenize(sentence)
    print(f"{label}")
    print(f"  {pieces}")
    print(f"  -> {len(pieces)} tokens")
    print()


### Code walkthrough — reading the three outputs

Loading three tokenizers and calling `.tokenize(sentence)` reuses the section 3 interface exactly; only the
checkpoint string changes.

- **GPT-2** marks a leading space as **`Ġ`**, attached to the *front* of a piece (`Ġtransformer`), because
  it was trained on raw web text where whitespace must survive the round trip. section 6 explains where
  that character comes from.
- **BERT's WordPiece** prefixes every piece after the first in a word with `##` — "glue me to the previous
  piece, no space".
- **XLNet's Unigram/SentencePiece** prefixes a `▁` (U+2581, not an underscore) to mark the *start* of a
  word — the opposite convention to WordPiece: SentencePiece marks word starts, WordPiece marks
  continuations.

No convention is more correct than another; each solves the same problem of making tokenization losslessly
reversible, whitespace included. What matters is that **you never mix one tokenizer's decode convention
with another's ids**.

## 6. Byte-level BPE: how GPT-2/GPT-4 avoid unknown tokens entirely

Character-level BPE, as described in section 4, can still hit an unknown symbol at inference time:
if it merges over Unicode *characters* and someone sends an emoji or a rare script your training corpus
never contained, the algorithm has no rule for it.

**Byte-level BPE** sidesteps this completely with one change: run the *exact same* merge algorithm over
the raw **UTF-8 bytes** of the text instead of over Unicode characters. There are only 256 possible byte
values, so the base vocabulary (before any merges) is fixed at 256 symbols and is guaranteed to cover
*any* text in *any* language or script, including emoji — because everything is bytes underneath. This is
the trick GPT-2, GPT-3/4, and most modern open LLMs (Llama, Qwen, Mistral) use.


In [ ]:
from transformers import AutoTokenizer

gpt2_tok = AutoTokenizer.from_pretrained("gpt2")

samples = [
    "Hello, world!",
    "Tokenization 🤯 works on emoji too.",
    "日本語のテキストも処理できます.",  # Japanese: "Japanese text can also be processed."
]

for text in samples:
    tokens = gpt2_tok.tokenize(text)
    ids = gpt2_tok.encode(text)
    print(f"{text!r}")
    print(f"  tokens ({len(tokens)}): {tokens}")
    print(f"  ids: {ids}")
    print(f"  round-trip decode: {gpt2_tok.decode(ids)!r}")
    print()


### Code walkthrough — byte-level tokens and the `Ġ` convention

**`.encode(text)`** is tokenize + convert in one call, returning plain ints. Unlike `tokenizer(text)` it
skips building a full `BatchEncoding`, which is handy when you only want to count tokens.

Look at the emoji and Japanese examples and you will see pieces that are not readable text at all. **These
are not Unicode characters** — they are individual UTF-8 **bytes**, each remapped to a printable stand-in
so that every byte value is a visible symbol (raw bytes 0–31 and 127 upward are not safely printable). The
🤯 emoji is 4 bytes in UTF-8, so if it never earned its own merge it appears as up to 4 byte-tokens: less
efficient than a common English word, but never an error.

**`Ġ`** is that same remapping applied to the space character, `0x20`. GPT-2's pre-tokenizer attaches the
preceding space to the *front* of the next word before merging, so `" world"` is its own trainable unit,
distinct from `"world"`. This is why GPT-2 tokenization is sensitive to leading whitespace — see pitfall 1
in section 10.

`decode(encode(text)) == text` holding exactly, even for emoji and Japanese, is the payoff: byte-level BPE
is lossless by construction, because every possible byte sequence is representable. Character-level
vocabularies must instead drop or `[UNK]` whatever they have not seen.

In [ ]:
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4")
text = "Tokenization 🤯 works on emoji too."
ids = enc.encode(text)
print("gpt-4 (tiktoken) ids   :", ids)
print("gpt-4 (tiktoken) tokens:", [enc.decode([i]) for i in ids])
print("gpt-4 (tiktoken) count :", len(ids))


### Code walkthrough — `tiktoken`

`tiktoken` is a standalone, dependency-light implementation of the same byte-level BPE family, used to
count tokens for OpenAI models without `transformers` or a trip to a model repo.

- **`encoding_for_model("gpt-4")`** looks up which named encoding (e.g. `cl100k_base`) that model uses and
  loads its merges and byte vocabulary.
- **`enc.encode(text)`** returns ids directly; there is no separate `.tokenize()`, since counting rather
  than linguistic analysis is the library's purpose.
- **`enc.decode([i])`**, called once per id, recovers each token's printable text — `tiktoken` exposes no
  `convert_ids_to_tokens` equivalent.

The practical point: **token count, not character or word count, is what APIs bill and what context
windows measure.** `len(enc.encode(text))` is the standard way to price a prompt before sending it.

### Exercise 6.1

Pick a sentence in a non-Latin script you know (Persian, Arabic, Chinese, etc.), or
reuse the Japanese example above. Compare `len(gpt2_tok.encode(sentence))` against
`len(gpt2_tok.encode(english_translation))` for a same-meaning English sentence of similar length. Which
one uses more tokens per word of meaning, and why does that matter for anyone paying per-token for API
access, or for a model's effective context window, in a non-English deployment?


## 7. Training your own tokenizer with the `tokenizers` library

Section 4 described the algorithm. The `tokenizers` library runs it for real — production-grade and
Rust-fast, on a corpus of any size. We'll train a small BPE tokenizer on a
Hugging Face dataset and then load it back through `AutoTokenizer`/`PreTrainedTokenizerFast` so it's
usable exactly like any pretrained tokenizer from the Hub.


In [ ]:
from datasets import load_dataset

raw = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train[:2000]")
raw = raw.filter(lambda ex: len(ex["text"].strip()) > 0)
print(raw)
print(raw[10])


### Code walkthrough — a small real corpus

- **`load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train[:2000]")`** reuses the slicing
  pattern from tutorial 0. WikiText-2 is a modest Wikipedia-derived corpus, and 2,000 rows is enough to see
  real merge behaviour without a long wait.
- **`.filter(lambda ex: len(ex["text"].strip()) > 0)`** drops rows that are empty once stripped. WikiText
  ships with many blank rows that were paragraph separators in the original format.

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=8000,
    min_frequency=2,
    special_tokens=["[UNK]", "[PAD]", "[BOS]", "[EOS]"],
    show_progress=True,
)

def batch_iterator(dataset, batch_size=1000):
    for i in range(0, len(dataset), batch_size):
        yield dataset[i : i + batch_size]["text"]

tokenizer.train_from_iterator(batch_iterator(raw), trainer=trainer)
print("Trained vocabulary size:", tokenizer.get_vocab_size())


### Code walkthrough — `Tokenizer`, `BpeTrainer`, and `train_from_iterator`

**`models.BPE(unk_token="[UNK]")`** picks BPE as the core algorithm. The library also offers
`models.WordPiece` and `models.Unigram`, and swapping that one line changes the algorithm family while the
rest of the pipeline stays identical. The `unk_token` is the fallback for something unrepresentable — with
byte-level pre-tokenization it essentially never fires, but the model still requires the slot.

**`pre_tokenizers.ByteLevel(add_prefix_space=False)`** runs *before* merging and fixes the initial split
points: split on whitespace boundaries, remap every byte to a printable stand-in. This is section 6's
scheme, now a configurable component rather than something baked into a checkpoint. **`add_prefix_space`**
controls whether a space is inserted before text that does not start with one, deciding whether the very
first word is treated as word-initial like every later word. GPT-2 uses `False` for single texts; `True` is
common when feeding a list of already-split words.

**`decoders.ByteLevel()`** must match the pre-tokenizer so ids reverse back through the same byte mapping.
Forgetting a matching decoder is a common bug that garbles output even when encoding is fine.

The trainer's arguments:

- **`vocab_size=8000`** — the target size, at which training stops. This is the most consequential
  hyperparameter in tokenizer training, chosen once per model family and never changed afterwards, since
  changing it changes the meaning of every id.
- **`min_frequency=2`** — a merge must occur at least this often to be accepted, so the vocabulary is not
  spent on typos and one-off noise.
- **`special_tokens=[...]`** — symbols guaranteed a slot regardless of corpus frequency. This reserves
  them; assigning their *roles* happens later.
- **`show_progress=True`** — cosmetic.

**`train_from_iterator`** takes an iterable of strings rather than the whole corpus in memory, and
`batch_iterator` yields 1,000-row slices, which is what lets the same code scale to corpora too large to
materialise. Passing **`trainer=trainer`** keeps architecture (`models.BPE`) and hyperparameters
(`BpeTrainer`) as separate objects, so the trainer works whichever model you chose.

In [ ]:
sample = "The tokenizer's vocabulary determines what unbelievably efficient models can learn."
out = tokenizer.encode(sample)
print("tokens:", out.tokens)
print("ids   :", out.ids)
print("decode:", tokenizer.decode(out.ids))


### Code walkthrough — using the trained tokenizer

**`tokenizer.encode(sample)`** on a raw `tokenizers.Tokenizer` returns an `Encoding` object exposing
`.tokens`, `.ids`, and `.attention_mask` as *attributes*, not dictionary keys — a small API difference
worth remembering when dropping to this lower level.

Compare this tokenizer's split of `"unbelievably"` with GPT-2's from section 6. Both are byte-level BPE, so
the mechanism is identical; the merges differ because this one saw 2,000 rows of WikiText and GPT-2 saw
web-scale text. **Vocabulary content is a direct fingerprint of training data.**

In [ ]:
from transformers import PreTrainedTokenizerFast

fast_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    unk_token="[UNK]",
    pad_token="[PAD]",
    bos_token="[BOS]",
    eos_token="[EOS]",
)

batch = fast_tokenizer(
    ["Short text.", "A somewhat longer sentence to force padding to kick in."],
    padding=True,
    return_tensors="pt",
)
print(batch["input_ids"])
print(batch["attention_mask"])


### Code walkthrough — wrapping it as a Hugging Face tokenizer

**`PreTrainedTokenizerFast`** is the class every fast `AutoTokenizer` actually instantiates. Here it is
built directly from **`tokenizer_object=tokenizer`** — our trained engine — instead of from a Hub
repository.

**`unk_token=`, `pad_token=`, `bos_token=`, `eos_token=`** tell `transformers` which special-token
*strings* play which *roles*. This is a distinct step from `special_tokens=[...]` in the trainer: that
reserved the vocabulary slots, this assigns meaning, so that `pad_token_id`, padding logic, and generation
code know which id to reach for.

The result supports the same `__call__` interface as any pretrained tokenizer, because it shares the same
base class. Saving with `save_pretrained("my-tokenizer")` and reloading with
`AutoTokenizer.from_pretrained("my-tokenizer")` is how you would keep a tokenizer trained for a new
language or domain.

## 8. The practical interface: `padding`, `truncation`, `max_length`, `return_tensors`

Real training and inference code always tokenizes **batches**, not single strings, and batches need
uniform-length tensors. This section reads every argument that controls that shape.


In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("bert-base-uncased")

batch_texts = [
    "Short.",
    "A medium length sentence for the batch.",
    "This is a considerably longer sentence, included specifically to force truncation once we set a small max_length.",
]

encoded = tok(
    batch_texts,
    padding=True,
    truncation=True,
    max_length=12,
    return_tensors="pt",
)

print("input_ids shape:", encoded["input_ids"].shape)
for row in encoded["input_ids"]:
    print(" ", row.tolist(), "->", tok.convert_ids_to_tokens(row.tolist()))
print()
print("attention_mask:")
for row in encoded["attention_mask"]:
    print(" ", row.tolist())


### Code walkthrough — `padding`, `truncation`, `max_length`, `return_tensors`

- **a list as the first argument** switches the tokenizer into batch mode: every field becomes a
  list-of-lists, or a 2D tensor when `return_tensors` is set, one row per input.
- **`padding=True`** pads every sequence to the **longest in this batch** (same as `padding="longest"`) by
  appending `pad_token_id` — `[PAD]`, id `0`, for BERT. Tensors must be rectangular, but padded positions
  carry no content.
- **`truncation=True`** cuts anything longer than `max_length` down to it, from the end, *before* padding
  applies. That is what clips the deliberately long third sentence.
- **`max_length=12`** is the ceiling for truncation and, with `padding="max_length"`, the padding target.
  Twelve is tiny on purpose so truncation is visible; a real value matches the model's context window — 512
  for base BERT, tens of thousands for modern LLMs.
- **`return_tensors="pt"`** returns PyTorch tensors. Alternatives are `"tf"`, `"np"`, or omitting it for
  plain Python lists, which is usually what you want while inspecting.

- **`return_attention_mask`** is not passed here because it defaults to on for any model that uses a mask,
  which is why one appears unasked. Set it to `False` only when a downstream consumer wants ids alone.

**Reading the output:** padded positions show id `0`, decode to `[PAD]`, and have `attention_mask` entries
of `0` at exactly those positions — so the model's attention ignores them and padding never influences the
predictions for shorter sequences sharing the batch.

### Before you run

`padding="max_length"` pads to the fixed `max_length` you pass, whatever the batch contains.

In this particular case nothing changes: `max_length` is 12 and the longest sequence in the batch already
reaches 12, so both settings produce the same shape. `"Short."` is padded to 12 either way. The difference
would show with a batch whose longest sequence were, say, 4 tokens — `padding=True` would stop at 4 while
`padding="max_length"` would still pad all the way to 12.

In [ ]:
encoded_fixed = tok(
    batch_texts,
    padding="max_length",
    truncation=True,
    max_length=12,
    return_tensors="pt",
)
print("input_ids shape:", encoded_fixed["input_ids"].shape)
print(encoded_fixed["input_ids"][0].tolist(), "  (row for the shortest sentence)")


### Code walkthrough — `longest` vs `max_length`

- **`padding=True`** (`"longest"`) pads to the longest sequence in *this* batch. Efficient, but the padded
  length varies between batches.
- **`padding="max_length"`** always pads to the fixed value. Every batch gets an identical shape, which
  some deployment settings require — compiled or exported models, and fixed-shape hardware such as certain
  TPU configurations — at the cost of computing over padding whenever inputs are much shorter.

## 9. Special tokens for chat models: `apply_chat_template`

Instruction-tuned / chat LLMs (Qwen, Llama-Instruct, ChatGPT-style models) are trained on conversations
wrapped in a specific format of role markers and special tokens — sending them plain, unformatted text
is a very common source of poor-quality output, because the model has essentially never seen unformatted
input during training. `apply_chat_template` builds the correct format for you from a list of role/content
dictionaries, without you needing to memorize each model family's exact markup.


In [ ]:
from transformers import AutoTokenizer

chat_tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")

messages = [
    {"role": "system", "content": "You are a concise assistant for an NLP course."},
    {"role": "user", "content": "In one sentence, what is byte-level BPE?"},
]

prompt_string = chat_tok.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
print(prompt_string)


### Code walkthrough — `apply_chat_template`

A conversation is a **list of dicts**, each with a **`role`** (`"system"`, `"user"`, `"assistant"`) and
**`content`**. That structure is shared across essentially every chat-tuned model in `transformers`, even
though the markup each produces differs.

- **`tokenize=False`** returns the formatted **string** rather than ids, which is what lets us print and
  inspect the markup here. Real inference code passes `tokenize=True` and feeds the ids straight to
  `model.generate(...)`.
- **`add_generation_prompt=True`** appends the marker meaning "your turn now, as the assistant". Without
  it the prompt ends after the user's turn and the model is liable to continue writing *another user
  turn* instead of answering — a subtle and common integration bug.

Each model family stores its own **chat template**, a Jinja2 template in the tokenizer config, which
decides the exact role markers and special tokens. That is why loading the matching tokenizer matters even
more for chat models than usual: the template is part of what the model was fine-tuned to expect.

## 10. Common tokenization pitfalls in real applications

These four issues account for a large share of "the model gives worse answers than it should" bug
reports in practice.


In [ ]:
gpt2_tok = AutoTokenizer.from_pretrained("gpt2")

print("with leading space   :", gpt2_tok.tokenize(" world"))
print("without leading space:", gpt2_tok.tokenize("world"))
print()
print("digits, glued  :", gpt2_tok.tokenize("3.14159"))
print("digits, spaced :", gpt2_tok.tokenize("3 . 1 4 1 5 9"))


### Code walkthrough — pitfalls 1 and 2

**Leading-space sensitivity.** `" world"` and `"world"` get **different ids** from GPT-2-family BPE,
because the leading space is fused onto the front of the word before merges apply. This catches people
building prompts by concatenation: `prompt + " " + continuation` and `prompt + continuation` produce
meaningfully different token sequences, though a human reader sees only an obviously-needed space.

**Digit splitting.** Byte-level BPE splits multi-digit numbers inconsistently — some short common numbers
get their own token while longer ones fragment digit-by-digit or in uneven groups. Try `"12345"` against
`"54321"` and the splits often differ arbitrarily. This is a well-documented reason LLMs have historically
been unreliable at multi-digit arithmetic: the model never sees a consistent ones/tens/hundreds
representation, only whatever chunks training frequency produced. Some newer tokenizers, Llama's among
them, force single-digit tokenization for exactly this reason.

In [ ]:
bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")
gpt2_tok = AutoTokenizer.from_pretrained("gpt2")

text = "Tokenization mismatch example."
bert_ids = bert_tok.encode(text)
print("BERT ids           :", bert_ids)
print("Decoded with GPT-2 :", gpt2_tok.decode(bert_ids))


### Code walkthrough — pitfall 3: tokenizer/model mismatch

Feeding BERT's ids to GPT-2's `.decode()` "works", in that it raises no error — every in-range integer is a
valid key into *some* vocabulary entry — but the output is meaningless, because id `2000` names a
completely different token in each vocabulary.

This is the rule to internalise from the whole notebook: **a tokenizer and a model must come from the same
checkpoint.** `AutoTokenizer.from_pretrained("bert-base-uncased")` with `AutoModel.from_pretrained("gpt2")`
is a bug that silently produces nonsense instead of crashing, which makes it far more dangerous than one
that fails loudly.

In [ ]:
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4")

pairs = [
    ("English", "The quick brown fox jumps over the lazy dog."),
    ("Persian", "روباه قهوه‌ای سریع از روی سگ تنبل می‌پرد."),
]

for lang, text in pairs:
    n = len(enc.encode(text))
    print(f"{lang:8s}: {n:3d} tokens for {len(text):3d} characters  ({n/len(text):.2f} tokens/char)")


### Code walkthrough — pitfall 4: multilingual token inefficiency

Most widely-used vocabularies are trained on corpora dominated by English and other high-resource,
Latin-script languages, because that is what the web holds. Text in other scripts therefore falls back to
smaller byte-level fragments, and the tokens-per-character ratio printed above runs noticeably higher.

Two concrete consequences. **Cost**: API pricing is per token, so the same sentence can cost several times
more in one language than another. **Effective context**: against an 8,000-token limit, a language needing
3× the tokens gets roughly a third of the usable window for the same content. Closing that gap is an active
area of tokenizer research.

**Pulling the four together.** Leading spaces, digit splitting, mismatched checkpoints, and multilingual
cost are not four unrelated bugs. Each one follows from the same fact this notebook opened with: the
tokenizer is a **separate trained artifact**, carrying its own frozen decisions about what counts as a
unit of text. Read its output when a model underperforms, before you blame the weights.


## Glossary — quick reference

| Term | Meaning |
|---|---|
| **Token** | The smallest unit a model reads/writes — a whole word, a subword piece, or a byte, depending on the tokenizer. |
| **Vocabulary** | The fixed set of all token strings a tokenizer can produce, each mapped to an integer id. |
| **BPE (Byte-Pair Encoding)** | Subword algorithm that iteratively merges the most *frequent* adjacent symbol pair. |
| **WordPiece** | Subword algorithm (BERT) that merges the pair maximizing training-data likelihood, not raw frequency. |
| **Unigram / SentencePiece** | Subword algorithm (T5, Llama) that starts from a large vocabulary and *prunes* down to the target size. |
| **Byte-level BPE** | BPE run over raw UTF-8 bytes instead of characters — guarantees no out-of-vocabulary input. |
| **OOV (out-of-vocabulary)** | A word-level tokenizer's failure mode: an input word has no vocabulary entry at all. |
| **Special tokens** | Reserved symbols with a role beyond plain text, e.g. `[CLS]`, `[SEP]`, `[PAD]`, `<|endoftext|>`, `[BOS]`/`[EOS]`. |
| **`attention_mask`** | Marks real-content positions (`1`) vs. padding positions (`0`) so padding never affects real predictions. |
| **`token_type_ids`** | Marks which of two packed segments (e.g. question vs. context) each token belongs to. |
| **Padding** | Extending shorter sequences in a batch with a pad token so all rows share one rectangular tensor shape. |
| **Truncation** | Cutting sequences longer than `max_length` down to that length before (or instead of) padding. |
| **Chat template** | A per-model format (role markers + special tokens) that `apply_chat_template` builds from a list of `{role, content}` messages. |
| **Tokenizer/model mismatch** | Using a tokenizer from a different checkpoint than the model — ids decode without error but are semantically meaningless. |

---
